# Synchrony Financial Call Center Forecasting - August 2025
Predicts 30-min interval-level calls, abandon rate, and CCT for 4 centers (A, B, C, D) across all 1,488 intervals in August.
# NOTE:
1. Slot: one 30-min interval (48 slots per day, 0=00:00, 47=23:30)
2. DoW : day of week (0 is Monday, 6 is Sunday)
3. Interval data (Apr–Jun 2025): used for model training and validation
4. Daily data (Aug 2025 actuals): used as anchor for volume prediction
5. Call volume and CCT are NOT predicted by LightGBM - they are computed from historical slot proportions× known daily totals
6. Only abandon rate and abandoned calls come from the model directly


# Step 1: Load the merged interval data
Imports all libraries and sets up two utility functions: one to load CSVs from S3 and one to save them back. Also initializes the US holiday calendar for 2025.

In [17]:
import pandas as pd
import numpy as np
import boto3
import io
import gc
import holidays
from datetime import datetime, timedelta
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
from sklearn.base import clone

s3 = boto3.client('s3')
bucket = 'synchrony-callcenter'

def load_csv_from_s3(key):
    obj = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_csv(io.BytesIO(obj['Body'].read()))

def save_csv_to_s3(df, key):
    buf = io.StringIO()
    df.to_csv(buf, index=False)
    s3.put_object(Bucket=bucket, Key=key, Body=buf.getvalue())
    print(f"  Saved: s3://{bucket}/{key}")

us_holidays = holidays.US(years=[2025])

print("Libraries are in")

Libraries are in


# Step 2" Feature engineering - lag features
For each interval row in the training data, it looks back 1 week and 2 weeks to find the same slot on the same weekday and pulls the call volume, abandoned calls, abandon rate, and CCT from that prior week. This gives the model historical context -> "what happened at this time last week?". Any slots that don't have a prior week available get filled with the average for that slot+weekday combination.

In [18]:
def add_lag_features(df):
    df = df.copy()
    df['datetime'] = pd.to_datetime(df['datetime'])
    df = df.sort_values('datetime').reset_index(drop=True)

    # create a lookup dictionary: datetime to row values
    # for fast lookup of prior week slots
    dt_index = df.set_index('datetime')

    lag_cols = ['call_volume', 'abandoned_calls', 'abandon_rate', 'CCT']

    for weeks_back in [1, 2]:
        for col in lag_cols:
            lag_col_name = f'lag_{weeks_back}w_{col}'
            df[lag_col_name] = df['datetime'].apply(
                lambda dt: dt_index.loc[dt - timedelta(weeks=weeks_back), col]
                if (dt - timedelta(weeks=weeks_back)) in dt_index.index
                else np.nan
            )

    # fill remaining lag nulls with slot+weekday mean
    lag_feature_cols = [c for c in df.columns if c.startswith('lag_')]
    for col in lag_feature_cols:
        null_mask = df[col].isnull()
        if null_mask.sum() > 0:
            df.loc[null_mask, col] = df.loc[null_mask].apply(
                lambda row: df[
                    (df['slot_index'] == row['slot_index']) &
                    (df['day_of_week'] == row['day_of_week']) &
                    df[col].notna()
                ][col].mean(),
                axis=1
            )

    print(f"  Lag features added. Null check on lags:")
    print(df[lag_feature_cols].isnull().sum())
    return df

print("Lag feature function ready")

Lag feature function ready


# Step 3: Build August 2025 prediction rows
Define build_august_rows(): Generates all 1,488 rows for August 2025 (31 days × 48 slots). For each slot it assigns the date, day of week (dow), slot index (..), whether it's a weekend, whether it's a holiday, and the month. Then it merges in "slot-level volume statistics" from training data, and merges in the known August 2025 daily totals from daily_clean - things like daily call volume, CCT, service level, abandon rate, and staffing. These daily actuals are important for prediction.

"Slot-level volume statistics" mean:
1. slot_mean - the average call volume for each slot index across all days in the training data. For example, slot 29 (14:30) might average 350 calls across all days regardless of what day of the week it is.
2. slot_dow_mean - the average call volume for each slot index broken down by day of week. So slot 29 on a Monday might average 420 calls, while slot 29 on a Sunday averages 180 calls.

RATIONALE: These become the features "slot_mean_volume" and "slot_dow_mean_volume" in FEATURE_COLS (upcoming). They give the model a baseline sense of how busy each time slot typically is - useful for predicting abandon rate and CCT, since a busier slot tends to have higher abandon rates and longer call times.

In [19]:
us_holidays = holidays.US(years=[2025])

def build_august_rows(center, slot_mean, slot_dow_mean):
    # generate all 30-min slots for August 2025
    aug_slots = pd.date_range(
        start='2025-08-01 00:00:00',
        end='2025-08-31 23:30:00',
        freq='30min'
    )
    aug_df = pd.DataFrame({'datetime': aug_slots})
    aug_df['date'] = aug_df['datetime'].dt.normalize()
    aug_df['day_of_week'] = aug_df['datetime'].dt.dayofweek
    aug_df['slot_index'] = aug_df['datetime'].dt.hour * 2 + aug_df['datetime'].dt.minute // 30
    aug_df['is_weekend'] = (aug_df['day_of_week'] >= 5).astype(int)
    aug_df['is_holiday'] = aug_df['date'].dt.date.apply(lambda d: d in us_holidays).astype(int)
    aug_df['month'] = 8
    # join slot mean features computed from training data
    # pass them in as arguments
    aug_df = aug_df.merge(slot_mean, on='slot_index', how='left')
    aug_df = aug_df.merge(slot_dow_mean, on=['slot_index', 'day_of_week'], how='left')

    # load daily clean for this center - contains daily context + staffing for augg
    daily = load_csv_from_s3(f'processed/center_{center}/daily_clean_{center}.csv')
    daily['date'] = pd.to_datetime(daily['date'])

    daily_cols = ['date', 'call_volume', 'CCT', 'service_level','abandon_rate', 'daily_staffing']
    daily_subset = daily[daily_cols].rename(columns={
        'call_volume': 'daily_call_volume',
        'CCT': 'daily_CCT',
        'service_level': 'daily_service_level',
        'abandon_rate':'daily_abandon_rate'
    })
    aug_df = aug_df.merge(daily_subset, on='date', how='left')
    aug_df['staffing_ratio'] = aug_df['daily_call_volume'] / aug_df['daily_staffing'].replace(0, np.nan)

    print(f"August rows built : {len(aug_df)}")
    print(f"Null check on daily:\n{aug_df[['daily_call_volume','daily_CCT','daily_service_level','daily_abandon_rate','daily_staffing']].isnull().sum()}")
    return aug_df

print("August row builder done")

August row builder done


# Step 4: Define features and targets
two lists: FEATURE_COLS (19 features the model sees) and TARGET_COLS (4 things the model predicts). Features include time-based signals like slot index and day of week, daily context like staffing and service level, slot-level historical averages, and lag features from the prior two weeks.

In [20]:
FEATURE_COLS = [
    'slot_index',
    'day_of_week',
    'is_weekend',
    'is_holiday',
    'month', 
    'slot_mean_volume',        #
    'slot_dow_mean_volume',    #
    'daily_call_volume',
    'daily_CCT',
    'daily_service_level',
    'daily_abandon_rate',
    'daily_staffing',
    'lag_1w_call_volume',
    'lag_2w_call_volume',
    'lag_1w_abandoned_calls',
    'lag_1w_abandon_rate',
    'lag_1w_CCT',
    'lag_2w_CCT',
    'staffing_ratio'
]

TARGET_COLS = [
    'call_volume',
    'abandoned_calls',
    'abandon_rate',
    'CCT'
]

print(f"Features : {len(FEATURE_COLS)} columns")
print(f"Targets  : {TARGET_COLS}")

Features : 19 columns
Targets  : ['call_volume', 'abandoned_calls', 'abandon_rate', 'CCT']


# Step 5: Train / validation split
train_val_split(): Splits chronologically (it's time series data, so I cannot be random about it): everything before June 16 is training data (April 1 to June 15'2025), everything from June 16 onwards is validation (June 16 to June 30). 
RATIONALE: This mimics real-world forecasting where you train on the past and validate on the most recent period.

In [21]:
def train_val_split(df):
    df = df.copy()
    df['datetime'] = pd.to_datetime(df['datetime'])

    # chronological split - train: Apr 1 – Jun 15, val: Jun 16 – Jun 30
    train_mask = df['datetime'] < '2025-06-16'
    val_mask = df['datetime'] >= '2025-06-16'

    train = df[train_mask].reset_index(drop=True)
    val = df[val_mask].reset_index(drop=True)

    print(f"Train : {len(train)} rows ({train['datetime'].min().date()} to {train['datetime'].max().date()})")
    print(f"Val: {len(val)} rows ({val['datetime'].min().date()} to {val['datetime'].max().date()})")
    return train, val

print("Train/val split function done")

Train/val split function done


# Step 6: Train LightGBM with MultiOutputRegressor
Defines train_model() and retrain_full(): Both train four separate LightGBM models (one per target). 
1. Call volume gets a quantile regression model (predicting the 55th percentile instead of the mean, to account for the asymmetric scoring penalty where underpredicting is worse than overpredicting).
2. The other three targets - abandoned calls, abandon rate, and CCT - get standard mean regression models.
3. retrain_full() does the same thing but on all available data (April–June) instead of just the training split.

RATIONALE: Quantile regression model, predicting the 55th percentile: 
1. This is to penalize underprediction more than overprediction. meaning, if you predict 100 calls but actual is 120, you lose more points (in hackathon) than if you predict 120 but actual is 100. Standard mean regression minimizes it, it doesn't know that being too low is worse than being too high.
2. Quantile regression at alpha=0.55 (value is a guess: 0.50 is pure median and above that, the model predicts higher than median)addresses this. "Instead of predicting the average expected value, it predicts a value where - based on training data, the actual will fall below your prediction 55% of the time." 
3. Change of plans: call volume is completely overridden in postprocess (upcoming) anyway by the slot proportion approach. So the quantile model's call volume prediction never actually reaches the output/technically unused in the final prediction. I've kept it because it was part of the architecture that produced one of my best score, and the separate model structure (rather than MultiOutputRegressor) is helped - by letting each target be trained independently.

In [22]:
def train_model(train):
    X_train = train[FEATURE_COLS]
    y_train = train[TARGET_COLS]

    # use separate models per target - quantile only for call_volume
    # mean regression for abandon_rate and CCT (work well already)

    lgbm_quantile = lgb.LGBMRegressor(
        objective='quantile',
        alpha=0.55,              # predict 55th percentile
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        num_leaves=31,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbose=-1
    )

    lgbm_mean = lgb.LGBMRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        num_leaves=31,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbose=-1
    )

    # train one model per target
    models = {}
    for col in TARGET_COLS:
        if col == 'call_volume':
            m = clone(lgbm_quantile)
        else:
            m = clone(lgbm_mean)
        m.fit(X_train, y_train[col])
        models[col] = m

    print(f"Models trained on {len(X_train)} rows, {len(FEATURE_COLS)} features")
    return models


def retrain_full(df):
    X_full = df[FEATURE_COLS]
    y_full = df[TARGET_COLS]

    lgbm_quantile = lgb.LGBMRegressor(
        objective='quantile',
        alpha=0.55,
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        num_leaves=31,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbose=-1
    )

    lgbm_mean = lgb.LGBMRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        num_leaves=31,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbose=-1
    )

    models = {}
    for col in TARGET_COLS:
        if col == 'call_volume':
            m = clone(lgbm_quantile)
        else:
            m = clone(lgbm_mean)
        m.fit(X_full, y_full[col])
        models[col] = m

    print(f"Models retrained on full {len(X_full)} rows")
    return models

# Step 7: Evaluate on validation set
defines evaluate_model(): which computes RMSE and MAPE per target on the validation set, and predict_august() which runs the full-data retrained model on the uugust rows to generate raw predictions

In [23]:
def evaluate_model(models, val):
    X_val  = val[FEATURE_COLS]
    y_val  = val[TARGET_COLS]

    print(f"\n  {'Target':<20} {'RMSE':>10} {'MAPE':>10}")
    print(f"  {'-'*42}")
    for col in TARGET_COLS:
        pred = models[col].predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val[col], pred))
        mask = y_val[col] != 0
        mape = np.mean(np.abs(
            (y_val[col][mask] - pred[mask]) / y_val[col][mask]
        )) * 100
        print(f"  {col:<20} {rmse:>10.3f} {mape:>9.2f}%")

def predict_august(models, aug_df):
    X_aug = aug_df[FEATURE_COLS]
    preds = {}
    for col in TARGET_COLS:
        preds[f'pred_{col}'] = models[col].predict(X_aug)

    y_pred = pd.DataFrame(preds)
    result = pd.concat([aug_df[['datetime', 'date', 'slot_index',
                                 'day_of_week', 'is_weekend',
                                 'is_holiday', 'daily_staffing',
                                 'daily_call_volume']], y_pred], axis=1)
    print(f"August predictions generated: {len(result)} rows")
    return result

# Step 8: Slot proportions and CCT means
This is the most important cell for volume accuracy. For each center, it loads the interval training data, computes what fraction of the daily total each slot carries (e.g. the 14:00 slot carries 4.2% of Monday's calls), and summarizes this using a 40% trimmed mean per slot+weekday combination. The 40% trim removes the most extreme outlier days before averaging, making the proportions more robust. CCT is summarized the same way but with a 25% trim since CCT needs less smoothing.

RATIONALE for 40%, 25% trim:
1. simple mean/0% trim): baseline - used a plain average of slot proportions across all training days. Composite was 15.920694.
2. switched from mean to median (which is effectively a 50% trim - it ignores everything except the middle value). This improved the composite to 15.869518. The improvement showed that outlier days in the training data were distorting the mean proportions
3. Tried different trim% then tried increasing one and decreasing another before deciding on 40% and 25%

In [24]:
from scipy import stats

slot_proportions = {}
slot_cct_means = {}

for center in ['A', 'B', 'C', 'D']:
    df = load_csv_from_s3(
        f'processed/center_{center}/interval_merged_{center}.csv')
    df['day_of_week'] = df['day_of_week'].astype(int)
    df['slot_index'] = df['slot_index'].astype(int)
    df['date']= pd.to_datetime(df['date'])

    daily_vol = df.groupby('date')['call_volume'].transform('sum')
    df['slot_proportion'] = df['call_volume'] / daily_vol.replace(0, np.nan)

    # trimmed mean - drop top and bottom 20% before averaging
    prop = df.groupby(
        ['slot_index', 'day_of_week'])['slot_proportion'].apply(
        lambda x: stats.trim_mean(x.dropna(), 0.40)
    ).reset_index()
    prop.columns = ['slot_index', 'day_of_week', 'slot_proportion']
    prop = prop.drop_duplicates(
        subset=['slot_index', 'day_of_week']).reset_index(drop=True)

    cct = df.groupby(
        ['slot_index', 'day_of_week'])['CCT'].apply(
        lambda x: stats.trim_mean(x.dropna(), 0.25)
    ).reset_index()
    cct.columns = ['slot_index', 'day_of_week', 'slot_cct_mean']
    cct = cct.drop_duplicates(
        subset=['slot_index', 'day_of_week']).reset_index(drop=True)

    slot_proportions[center] = prop
    slot_cct_means[center] = cct
    print(f"Center {center} | prop: {prop.shape} | cct: {cct.shape}")

Center A | prop: (336, 3) | cct: (336, 3)
Center B | prop: (336, 3) | cct: (336, 3)
Center C | prop: (336, 3) | cct: (336, 3)
Center D | prop: (336, 3) | cct: (336, 3)


# Step 9: Post-process predictions
postprocess(): This is where the raw model predictions get overridden. Call volume and CCT are not predicted by LightGBM. LightGBM trains on them, but those predictions get thrown away in postprocess and replaced by:
1. Call volume = slot proportion × known daily total × 1.05
2. CCT = trimmed mean CCT for that slot+weekday combination, capped at 600
Only abandon rate and abandoned calls flow through from the model's raw predictions, clipped to valid ranges.

RATIONALE:
1. Why 1.05 (refer Exploratory section at the end of this notebook): I tried 1.00 to 1.05 values. Although the actual and predictions were close when the multiplier was 1.00, it did not end up giving high composite score upon submission. Among the options tried, 1.05 was the best score producing multiplier.

2. Why 600 seconds for CCT cap?
600 seconds = 10 minutes. This is a standard upper bound for average call handling time in a call center context - calls rarely average more than 10 minutes across a 30-minute slot. (refer Exploratory section for data and details)

In [25]:
def postprocess(df, slot_prop, slot_cct):
    df = df.copy()
    df['datetime']= pd.to_datetime(df['datetime'])
    df['day']= df['datetime'].dt.date
    df['day_of_week'] = df['datetime'].dt.dayofweek.astype(int)
    df['slot_index']= df['slot_index'].astype(int)

    df = df.merge(slot_prop, on=['slot_index', 'day_of_week'], how='left')

    df['pred_call_volume'] = (
        df['slot_proportion'] * df['daily_call_volume'] * 1.05
    ).clip(lower=0).round().astype(int)

    df['pred_abandoned_calls'] = df['pred_abandoned_calls'].clip(
        lower=0).round().astype(int)
    df['pred_abandon_rate']= df['pred_abandon_rate'].clip(0.0, 1.0)

    df = df.merge(slot_cct, on=['slot_index', 'day_of_week'], how='left')
    df['pred_CCT'] = df['slot_cct_mean'].clip(lower=0, upper=600)

    df.drop(columns=['day', 'slot_proportion', 'slot_cct_mean'], inplace=True)

    print("  Post-processing complete.")
    print(df[['pred_call_volume', 'pred_abandoned_calls',
              'pred_abandon_rate', 'pred_CCT']].describe())
    return df

# Steps 10: Main loop
Runs everything for each of the four centers in sequence. For each center: loads interval data, computes slot features, adds lag features, builds August rows, trains the model, evaluates on validation, retrains on full data, predicts August, postprocesses, and saves to S3. Memory is cleared between centers to avoid running out of RAM.

In [26]:
for center in ['A', 'B', 'C', 'D']:
    print(f"\n{'='*60}")
    print(f"CENTER {center}")
    print('='*60)

    # Step 1: load and prepare
    df = load_csv_from_s3(
        f'processed/center_{center}/interval_merged_{center}.csv')
    df['datetime'] = pd.to_datetime(df['datetime'])
    df['month'] = df['datetime'].dt.month
    df['day_of_week'] = df['day_of_week'].astype(int)
    df['slot_index']= df['slot_index'].astype(int)
    print(f"  Loaded: {df.shape}")

    # Step 1b - slot mean features
    slot_mean = df.groupby(
        'slot_index')['call_volume'].mean().reset_index()
    slot_mean.rename(
        columns={'call_volume': 'slot_mean_volume'}, inplace=True)
    slot_dow_mean = df.groupby(
        ['slot_index', 'day_of_week'])['call_volume'].mean().reset_index()
    slot_dow_mean.rename(
        columns={'call_volume': 'slot_dow_mean_volume'}, inplace=True)
    df = df.merge(slot_mean,     on='slot_index',                  how='left')
    df = df.merge(slot_dow_mean, on=['slot_index', 'day_of_week'], how='left')
    print(f"  Slot mean features added.")
    print(f"  Nulls: {df[['slot_mean_volume','slot_dow_mean_volume']].isnull().sum().to_dict()}")

    df['staffing_ratio'] = df['daily_call_volume'] / df['daily_staffing'].replace(0, np.nan)
    # Step 2 - lag features
    print("\n-- Step 2: Lag features --")
    df = add_lag_features(df)

    # Step 3 - build august rows
    print("\n-- Step 3: Build August rows --")
    aug_df = build_august_rows(center, slot_mean, slot_dow_mean)

    # add lag features to august using last 2 weeks of june as proxy
    june_tail = df[pd.to_datetime(df['datetime']) >= '2025-06-17'].copy()
    for weeks_back in [1, 2]:
        for col in ['call_volume', 'abandoned_calls', 'abandon_rate', 'CCT']:
            lag_col = f'lag_{weeks_back}w_{col}'
            slot_dow_mean_lag = june_tail.groupby(
                ['slot_index', 'day_of_week'])[col].mean().reset_index()
            slot_dow_mean_lag.rename(columns={col: lag_col}, inplace=True)
            aug_df = aug_df.merge(
                slot_dow_mean_lag, on=['slot_index', 'day_of_week'], how='left')

    # Step 4 - already defined as constants above

    # Step 5 - split
    print("\n-- Step 5: Train/val split --")
    train, val = train_val_split(df)

    # Step 6 -train
    print("\n-- Step 6: Train model --")
    model = train_model(train)

    # Step 7- evaluate
    print("\n-- Step 7: Evaluate --")
    evaluate_model(model, val)

    # Step 8 - retrain on full data
    print("\n-- Step 8: Retrain on full data --")
    model_full = retrain_full(df)

    # Step 9 - predict august
    print("\n-- Step 9: Predict August --")
    aug_preds = predict_august(model_full, aug_df)

    # Step 10 - post-process
    print("\n-- Step 10: Post-process --")
    aug_preds = postprocess(aug_preds, slot_proportions[center], slot_cct_means[center])

    # Step 11 - save
    print("\n-- Step 11: Save to S3 --")
    save_csv_to_s3(
        aug_preds,
        f'outputs/predictions/predictions_{center}_aug2025.csv'
    )

    # Step 12 - clear memory
    del df, train, val, model, model_full, aug_df, aug_preds
    gc.collect()
    print(f"\n  Memory cleared. Center {center} complete.")

print("\nAll centers completed")


CENTER A
  Loaded: (4076, 18)
  Slot mean features added.
  Nulls: {'slot_mean_volume': 0, 'slot_dow_mean_volume': 0}

-- Step 2: Lag features --
  Lag features added. Null check on lags:
lag_1w_call_volume         0
lag_1w_abandoned_calls     0
lag_1w_abandon_rate        0
lag_1w_CCT                 0
lag_2w_call_volume        16
lag_2w_abandoned_calls    16
lag_2w_abandon_rate       16
lag_2w_CCT                16
dtype: int64

-- Step 3: Build August rows --
August rows built    : 1488
Null check on daily  :
daily_call_volume      0
daily_CCT              0
daily_service_level    0
daily_abandon_rate     0
daily_staffing         0
dtype: int64

-- Step 5: Train/val split --
Train : 3401 rows (2025-04-01 to 2025-06-15)
Val: 675 rows (2025-06-16 to 2025-06-30)

-- Step 6: Train model --
Models trained on 3401 rows, 19 features

-- Step 7: Evaluate --

  Target                     RMSE       MAPE
  ------------------------------------------
  call_volume              12.573     26.08%

# Step 11: Creating deliverable
Loads the four per-center prediction files from S3, combines them into the required 1488×19 submission format, populates Month/Day/Interval columns, applies two data quality fixes (clips call volume minimum to 1, ensures abandoned calls never exceed calls offered), and saves the final submission file to S3.

In [27]:
pred_dfs = {}
for center in ['A', 'B', 'C', 'D']:
    df = load_csv_from_s3(
        f'outputs/predictions/predictions_{center}_aug2025.csv')
    df['datetime'] = pd.to_datetime(df['datetime'])
    pred_dfs[center] = df

base = pred_dfs['A'][['datetime']].copy()

# populate Month, Day, Interval from datetime
base['Month']= base['datetime'].dt.strftime('%B')
base['Day']= base['datetime'].dt.day
base['Interval'] = (base['datetime'].dt.hour.astype(str) + ':' +
                    base['datetime'].dt.minute.astype(str).str.zfill(2))

for center in ['A', 'B', 'C', 'D']:
    df = pred_dfs[center]

    # fix 1 - clip call volume minimum to 1
    df['pred_call_volume'] = df['pred_call_volume'].clip(lower=1)

    # fix 2 - abandoned calls cannot exceed calls offered
    df['pred_abandoned_calls'] = np.minimum(
        df['pred_abandoned_calls'].clip(lower=0),
        df['pred_call_volume']
    )

    base[f'Calls_Offered_{center}']= df['pred_call_volume'].values
    base[f'Abandoned_Calls_{center}'] = df['pred_abandoned_calls'].values
    base[f'Abandoned_Rate_{center}']= df['pred_abandon_rate'].values
    base[f'CCT_{center}']  = df['pred_CCT'].values

base.drop(columns=['datetime'], inplace=True)

print(f"Submission file shape : {base.shape}")
print(f"Columns               : {list(base.columns)}")
print(f"\nFirst 3 rows:")
print(base.head(3).to_string())
print(f"\nLast 3 rows:")
print(base.tail(3).to_string())

save_csv_to_s3(base, 'outputs/predictions/final_submission.csv')
print("\nFinal submission file saved")

Submission file shape : (1488, 19)
Columns               : ['Month', 'Day', 'Interval', 'Calls_Offered_A', 'Abandoned_Calls_A', 'Abandoned_Rate_A', 'CCT_A', 'Calls_Offered_B', 'Abandoned_Calls_B', 'Abandoned_Rate_B', 'CCT_B', 'Calls_Offered_C', 'Abandoned_Calls_C', 'Abandoned_Rate_C', 'CCT_C', 'Calls_Offered_D', 'Abandoned_Calls_D', 'Abandoned_Rate_D', 'CCT_D']

First 3 rows:
    Month  Day Interval  Calls_Offered_A  Abandoned_Calls_A  Abandoned_Rate_A       CCT_A  Calls_Offered_B  Abandoned_Calls_B  Abandoned_Rate_B       CCT_B  Calls_Offered_C  Abandoned_Calls_C  Abandoned_Rate_C       CCT_C  Calls_Offered_D  Abandoned_Calls_D  Abandoned_Rate_D       CCT_D
0  August    1     0:00                6                  1          0.056192  343.167143               38                  0          0.014773  303.104286               89                  5          0.021879  351.938571               46                  1          0.008001  329.141429
1  August    1     0:30                5     

# Exploratory

# Justifying multiplier of 1.05

In [28]:
# show actual vs predicted daily totals for August 2025 - raw numbers
from scipy import stats

for center in ['A', 'B', 'C', 'D']:
    # load interval training data to compute slot proportions
    df = load_csv_from_s3(
        f'processed/center_{center}/interval_merged_{center}.csv')
    df['day_of_week'] = df['day_of_week'].astype(int)
    df['slot_index']  = df['slot_index'].astype(int)
    df['date']        = pd.to_datetime(df['date'])

    daily_vol = df.groupby('date')['call_volume'].transform('sum')
    df['slot_proportion'] = df['call_volume'] / daily_vol.replace(0, np.nan)

    prop = df.groupby(
        ['slot_index', 'day_of_week'])['slot_proportion'].apply(
        lambda x: stats.trim_mean(x.dropna(), 0.40)
    ).reset_index()
    prop.columns = ['slot_index', 'day_of_week', 'slot_proportion']

    # load august 2025 daily actuals
    daily = load_csv_from_s3(
        f'processed/center_{center}/daily_clean_{center}.csv')
    daily['date'] = pd.to_datetime(daily['date'])
    aug = daily[
        (daily['date'].dt.month == 8) &
        (daily['date'].dt.year == 2025)
    ].copy()

    # build august slots
    aug_slots = pd.date_range(
        start='2025-08-01 00:00:00',
        end='2025-08-31 23:30:00',
        freq='30min'
    )
    aug_df = pd.DataFrame({'datetime': aug_slots})
    aug_df['date']        = aug_df['datetime'].dt.normalize()
    aug_df['day_of_week'] = aug_df['datetime'].dt.dayofweek
    aug_df['slot_index']  = aug_df['datetime'].dt.hour * 2 + \
                             aug_df['datetime'].dt.minute // 30

    aug_df = aug_df.merge(prop, on=['slot_index', 'day_of_week'], how='left')
    aug_df = aug_df.merge(
        aug[['date', 'call_volume']].rename(
            columns={'call_volume': 'daily_actual'}),
        on='date', how='left'
    )

    # predicted daily totals with each multiplier
    aug_df['pred_1_00'] = aug_df['slot_proportion'] * aug_df['daily_actual'] * 1.00
    aug_df['pred_1_05'] = aug_df['slot_proportion'] * aug_df['daily_actual'] * 1.05

    summary = aug_df.groupby('date').agg(
        actual        =('daily_actual', 'first'),
        pred_mult_100 =('pred_1_00', 'sum'),
        pred_mult_105 =('pred_1_05', 'sum')
    ).reset_index()
    summary['date'] = summary['date'].dt.strftime('%Y-%m-%d')

    print(f"\nCenter {center}:")
    print(f"{'Date':<12} {'Actual':>10} {'Pred@1.00':>12} {'Pred@1.05':>12}")
    print('-' * 48)
    for _, row in summary.iterrows():
        print(f"{row['date']:<12} {row['actual']:>10.0f} "
              f"{row['pred_mult_100']:>12.0f} {row['pred_mult_105']:>12.0f}")
    print(f"{'MEAN':<12} {summary['actual'].mean():>10.0f} "
          f"{summary['pred_mult_100'].mean():>12.0f} "
          f"{summary['pred_mult_105'].mean():>12.0f}")


Center A:
Date             Actual    Pred@1.00    Pred@1.05
------------------------------------------------
2025-08-01         5011         4981         5230
2025-08-02         3409         3416         3587
2025-08-03         1477         1475         1549
2025-08-04         5076         5056         5308
2025-08-05         4888         4878         5122
2025-08-06         4058         4066         4269
2025-08-07         4321         4280         4494
2025-08-08         4521         4494         4718
2025-08-09         2820         2826         2967
2025-08-10         1280         1279         1342
2025-08-11         4869         4849         5092
2025-08-12         4011         4003         4203
2025-08-13         3999         4007         4207
2025-08-14         4137         4098         4303
2025-08-15         3831         3808         3998
2025-08-16         3048         3054         3207
2025-08-17         1425         1423         1495
2025-08-18         4298         4281    

# Justifying 600secs cap (refer code below):
The mean CCT across all centers is 314-323 seconds (~5.2-5.4 minutes) and the median is similar, meaning the typical slot has very reasonable handling times. The 95th percentile is 383–462 seconds (~6.4-7.7 minutes) - still well below 600.
Slots exceeding 600 seconds are rare - only 0.37% to 2.28% of all slots depending on center. And when you look at which slots they are, they're almost exclusively overnight/early morning slots (slot_index 2-12, meaning 1:00am-6:00am) where call volume is near zero. A single long call in a slot with 1-2 calls total inflates the average dramatically - for example Center D slot 8 shows a CCT of 4,786 seconds (80 minutes) which is clearly a outlier, not a real average handling time.

In [29]:
for center in ['A', 'B', 'C', 'D']:
    df = load_csv_from_s3(
        f'processed/center_{center}/interval_merged_{center}.csv')
    df['slot_index'] = df['slot_index'].astype(int)

    print(f"\nCenter {center} CCT distribution (seconds):")
    print(f"  mean   : {df['CCT'].mean():.1f}")
    print(f"  median : {df['CCT'].median():.1f}")
    print(f"  95th % : {df['CCT'].quantile(0.95):.1f}")
    print(f"  99th % : {df['CCT'].quantile(0.99):.1f}")
    print(f"  max    : {df['CCT'].max():.1f}")
    print(f"  >600   : {(df['CCT'] > 600).sum()} rows "
          f"({(df['CCT'] > 600).mean()*100:.2f}% of slots)")
    print(f"  >600 rows sample:")
    print(df[df['CCT'] > 600][['slot_index', 'CCT']].head(5).to_string())


Center A CCT distribution (seconds):
  mean   : 314.3
  median : 310.3
  95th % : 462.1
  99th % : 780.6
  max    : 2799.0
  >600   : 93 rows (2.28% of slots)
  >600 rows sample:
     slot_index      CCT
4             4   667.00
56           12  1085.00
184           2   716.67
185           3   676.50
187           5  1681.00

Center B CCT distribution (seconds):
  mean   : 323.4
  median : 323.0
  95th % : 435.8
  99th % : 726.8
  max    : 3270.0
  >600   : 77 rows (1.80% of slots)
  >600 rows sample:
     slot_index     CCT
54            6   906.5
186           2   988.5
236           4   601.0
383           8   692.0
429           7  1155.0

Center C CCT distribution (seconds):
  mean   : 323.4
  median : 329.6
  95th % : 383.4
  99th % : 512.3
  max    : 882.5
  >600   : 16 rows (0.37% of slots)
  >600 rows sample:
      slot_index     CCT
202           10  722.86
250           10  798.00
620            5  765.00
865           10  638.11
1100           5  754.00

Center D CCT dis